# Day 065 — Exercise 1: Metrics Middleware

Day 061 introduced `MetricsCollector` and `@app.middleware('http')`. Today we combine them into a reusable function: `add_metrics_middleware` takes any existing FastAPI app and instruments it — adding the middleware and a `/metrics` endpoint — then returns the same app.

This is the **decorator pattern**: enhance an object without modifying its original definition.

In [ ]:
import time
from fastapi import FastAPI
from starlette.testclient import TestClient

class MetricsCollector:
    """Records HTTP request metrics (from Day 061)."""
    def __init__(self):
        self._requests  = 0
        self._errors    = 0
        self._latencies: list[float] = []

    def record(self, status_code: int, duration_ms: float) -> None:
        self._requests += 1
        if status_code >= 400:
            self._errors += 1
        self._latencies.append(duration_ms)

    def summary(self) -> dict:
        avg  = sum(self._latencies)/len(self._latencies) if self._latencies else 0.0
        rate = self._errors/self._requests if self._requests else 0.0
        return {
            "requests":       self._requests,
            "errors":         self._errors,
            "avg_latency_ms": round(avg, 1),
            "error_rate":     round(rate, 3),
        }

    def reset(self):
        self._requests = 0; self._errors = 0; self._latencies.clear()


## Task

Implement `add_metrics_middleware(app, collector) -> FastAPI`:

1. Add `@app.middleware('http')` that times each request and calls `collector.record(status_code, duration_ms)`
2. Add `GET /metrics` → `collector.summary()`
3. Return the same `app` object

Timing: `start = time.monotonic()` → `await call_next(request)` → `duration = (time.monotonic() - start) * 1000`

## Your Implementation

In [ ]:
def add_metrics_middleware(app: FastAPI,
                           collector: MetricsCollector) -> FastAPI:
    """Add HTTP middleware and /metrics endpoint to an existing FastAPI app.

    The middleware:
    - Captures start time before each request (time.monotonic())
    - Calls next handler (await call_next(request))
    - Computes duration_ms = (time.monotonic() - start) * 1000
    - Records via collector.record(response.status_code, duration_ms)

    The /metrics endpoint:
    - GET /metrics → collector.summary()

    Returns the same app (mutates it).
    """
    # TODO: add @app.middleware('http') + GET /metrics route
    raise NotImplementedError


In [ ]:
def add_metrics_middleware(app: FastAPI,
                           collector: MetricsCollector) -> FastAPI:
    @app.middleware("http")
    async def _middleware(request, call_next):
        start    = time.monotonic()
        response = await call_next(request)
        duration = (time.monotonic() - start) * 1000
        collector.record(response.status_code, duration)
        return response

    @app.get("/metrics")
    def _metrics():
        return collector.summary()

    return app


## Automated checks

In [ ]:
score, total = 0, 5
try:
    from fastapi import FastAPI
    from starlette.testclient import TestClient
    from starlette.responses import JSONResponse

    # build a minimal app to instrument
    base_app  = FastAPI()
    collector = MetricsCollector()

    @base_app.get("/ping")
    def ping(): return {"pong": True}

    @base_app.get("/boom")
    def boom(): return JSONResponse({"err": "oops"}, status_code=500)

    add_metrics_middleware(base_app, collector)
    c = TestClient(base_app, raise_server_exceptions=False)

    # /metrics endpoint exists
    r = c.get("/metrics")
    assert r.status_code == 200
    score += 1; print("\u2705 GET /metrics endpoint exists and returns 200")

    # make 2 successful requests
    c.get("/ping"); c.get("/ping")
    m = c.get("/metrics").json()
    # middleware records /metrics too, so requests >= 2 (exact count depends on order)
    assert m["requests"] >= 2
    score += 1; print("\u2705 requests counter increments on each call")

    # trigger an error
    c.get("/boom")
    m2 = c.get("/metrics").json()
    assert m2["errors"] >= 1
    score += 1; print("\u2705 errors counter increments on 5xx responses")

    # avg_latency_ms is a float >= 0
    assert isinstance(m2["avg_latency_ms"], (int, float)) and m2["avg_latency_ms"] >= 0
    score += 1; print("\u2705 avg_latency_ms is a non-negative number")

    # error_rate between 0 and 1
    assert 0.0 <= m2["error_rate"] <= 1.0
    score += 1; print("\u2705 error_rate is between 0.0 and 1.0")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def add_metrics_middleware(app: FastAPI,
                           collector: MetricsCollector) -> FastAPI:
    @app.middleware("http")
    async def _middleware(request, call_next):
        start    = time.monotonic()
        response = await call_next(request)
        duration = (time.monotonic() - start) * 1000
        collector.record(response.status_code, duration)
        return response

    @app.get("/metrics")
    def _metrics():
        return collector.summary()

    return app
```

**Why mutate and return?** Returning `app` allows chaining: `add_monitoring(add_cors(FastAPI(), origins), collector)`. It also makes testing natural: `app = add_metrics_middleware(build_base(), c)`. Mutating is safe here because FastAPI registers middleware at definition time.

</details>